# Stg Layer — osm_restaurants (OpenStreetMap food places, full rebuild)

Source: `${bronze}.osm_pois_raw` (task `osm_ingest`, weekly Overpass snapshot). Added 2026-10-31. Data © OpenStreetMap contributors (ODbL).

1. Latest snapshot only (`MAX(ingest_ts)`), one row per `osm_key` (`node/123`, `way/456`, ...); elements without a name are dropped (they cannot be matched).
2. `restaurant_name_normalized` = same rule as `stg_places` / `stg_restaurant_mentions` (unaccented, lower case, letters and digits).
3. `cuisine` = the dashboard label of the first OSM `cuisine` value (`;`-separated) found in `OSM_CUISINES` (same labels as `gold/restaurants` `CUISINE_TYPES`); `diet:vegetarian|vegan = only` → Chay; `amenity` cafe / ice_cream without a cuisine → Cafe & tráng miệng; else NULL.
4. `opening_hours` kept as OSM writes it (e.g. `Mo-Su 06:00-22:00`); phone / website / street address when tagged.

`gold/restaurants` matches these places to `dim_restaurant` by distance (≤ 100 m) and name, and uses them only to fill gaps (cuisine when Google and the name say nothing, opening hours).

In [ ]:
%run ../gold/_udfs

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SOURCE = f"{BRONZE}.osm_pois_raw"
TARGET = f"{SILVER}.stg_osm_restaurants"
# OSM cuisine value -> dashboard cuisine label (labels of gold/restaurants CUISINE_TYPES). Order = priority.
OSM_CUISINES = [
    ("Món Việt", ["vietnamese", "pho", "banh_mi", "bun", "com_tam", "regional"]),
    ("Món Hoa", ["chinese", "cantonese", "dim_sum", "taiwanese", "sichuan", "hong_kong"]),
    ("Món Nhật", ["japanese", "sushi", "ramen", "udon", "yakiniku", "izakaya"]),
    ("Món Hàn", ["korean"]),
    ("Món Thái", ["thai"]),
    ("Món Ấn", ["indian"]),
    ("Món Âu / Mỹ", ["italian", "pizza", "french", "spanish", "greek", "mediterranean", "german", "british",
                     "european", "american", "steak_house", "burger", "mexican", "brazilian", "western"]),
    ("Món quốc tế khác", ["indonesian", "malaysian", "filipino", "middle_eastern", "lebanese", "turkish",
                          "african", "russian", "international"]),
    ("Hải sản", ["seafood", "fish"]),
    ("Lẩu & nướng", ["hot_pot", "barbecue", "bbq", "grill"]),
    ("Chay", ["vegetarian", "vegan"]),
    ("Buffet", ["buffet"]),
    ("Ăn nhanh", ["sandwich", "chicken", "fried_chicken", "fast_food", "kebab", "hot_dog"]),
    ("Cafe & tráng miệng", ["coffee_shop", "coffee", "tea", "bubble_tea", "ice_cream", "dessert", "cake",
                            "bakery", "juice", "donut", "crepe", "pastry"]),
    ("Món Á khác", ["asian"]),
]
spark.createDataFrame(
    [(v, label, prio) for prio, (label, values) in enumerate(OSM_CUISINES) for v in values],
    "osm_value STRING, cuisine STRING, priority INT",
).createOrReplaceTempView("osm_cuisines")
print(f"Bronze source: {SOURCE} · Silver target: {TARGET}")

In [ ]:
# --- Rebuild from the latest snapshot (empty table until the first Overpass answer reaches Bronze) ---
if spark.catalog.tableExists(SOURCE) and spark.table(SOURCE).limit(1).count() > 0:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {TARGET}
        COMMENT 'OpenStreetMap food places in Ho Chi Minh City (latest weekly snapshot, named only). Data © OpenStreetMap contributors, ODbL.'
        AS
        WITH snap AS (
            SELECT * FROM {SOURCE} WHERE ingest_ts = (SELECT MAX(ingest_ts) FROM {SOURCE})
        ),
        one AS (   -- one row per element (a repeat inside one snapshot is not expected)
            SELECT * FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY osm_type, osm_id ORDER BY ingest_ts DESC) AS rn FROM snap
            ) WHERE rn = 1
        ),
        named AS (
            SELECT *, COALESCE(tags['name:vi'], tags['name'], tags['name:en']) AS place_name FROM one
        ),
        cuisine_hit AS (   -- first known OSM cuisine value, by OSM_CUISINES priority then position in the tag
            SELECT v.osm_type, v.osm_id, min_by(c.cuisine, struct(c.priority, v.pos)) AS cuisine
            FROM (
                SELECT n.osm_type, n.osm_id, e.pos, e.osm_value
                FROM named n
                LATERAL VIEW posexplode(split(lower(replace(COALESCE(n.tags['cuisine'], ''), ' ', '')), ';')) e AS pos, osm_value
            ) v
            JOIN osm_cuisines c ON c.osm_value = v.osm_value
            GROUP BY v.osm_type, v.osm_id
        )
        SELECT
            concat(n.osm_type, '/', CAST(n.osm_id AS STRING))                          AS osm_key,
            n.osm_type,
            n.osm_id,
            n.tags['amenity']                                                          AS amenity,
            n.place_name,
            regexp_replace(lower(trim(unaccent_vi(normalize_keyword(n.place_name)))), '[^a-z0-9 ]', '') AS restaurant_name_normalized,
            n.tags['cuisine']                                                          AS cuisine_raw,
            CASE WHEN h.cuisine IS NOT NULL THEN h.cuisine
                 WHEN n.tags['diet:vegan'] = 'only' OR n.tags['diet:vegetarian'] = 'only' THEN 'Chay'
                 WHEN n.tags['amenity'] IN ('cafe', 'ice_cream') THEN 'Cafe & tráng miệng'
            END                                                                        AS cuisine,
            n.tags['opening_hours']                                                    AS opening_hours,
            COALESCE(n.tags['contact:phone'], n.tags['phone'])                         AS phone,
            COALESCE(n.tags['contact:website'], n.tags['website'])                     AS website,
            nullif(concat_ws(' ', n.tags['addr:housenumber'], n.tags['addr:street']), '') AS street_address,
            n.latitude,
            n.longitude,
            n.osm_base,
            n.ingest_ts,
            current_timestamp()                                                        AS etl_date
        FROM named n
        LEFT JOIN cuisine_hit h ON h.osm_type = n.osm_type AND h.osm_id = n.osm_id
        WHERE n.place_name IS NOT NULL AND trim(n.place_name) <> ''
    """)
else:
    print(f"{SOURCE} is empty or missing (no Overpass answer yet): {TARGET} stays empty")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TARGET} (
            osm_key STRING, osm_type STRING, osm_id BIGINT, amenity STRING, place_name STRING,
            restaurant_name_normalized STRING, cuisine_raw STRING, cuisine STRING, opening_hours STRING,
            phone STRING, website STRING, street_address STRING, latitude DOUBLE, longitude DOUBLE,
            osm_base TIMESTAMP, ingest_ts TIMESTAMP, etl_date TIMESTAMP
        ) USING DELTA
    """)

s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT osm_key) AS ids, COUNT(cuisine) AS with_cuisine,
           COUNT(opening_hours) AS with_hours, MAX(osm_base) AS osm_base
    FROM {TARGET}""").collect()[0]
print(f"✅ {TARGET}: {s['n']} named places · cuisine {s['with_cuisine']} · opening hours {s['with_hours']} · OSM data as of {s['osm_base']}")
print("📊 Grain check (osm_key): " + ("✅" if s["n"] == s["ids"] else "❌ duplicates"))
display(spark.sql(f"SELECT amenity, cuisine, COUNT(*) AS n FROM {TARGET} GROUP BY 1, 2 ORDER BY n DESC"))
print("\n--- stg_osm_restaurants complete ---")